fine tuning 

In [ ]:
#training
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

!pip install transformers datasets accelerate -q
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Data
# ============================================================
train_df = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/train/train_2100.csv')
val_df = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/validation/val_450.csv')
test_df = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/test/test_450.csv')

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(train_df['label'].value_counts())

# Label encoding
label2id = {'human': 0, 'ai_generated': 1, 'refine': 2}
id2label = {v: k for k, v in label2id.items()}

train_df['label_id'] = train_df['label'].map(label2id)
val_df['label_id'] = val_df['label'].map(label2id)
test_df['label_id'] = test_df['label'].map(label2id)

# Tokenizer + Dataset
# ============================================================
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

class ClaimDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = ClaimDataset(
    train_df['claim1'].tolist(),
    train_df['label_id'].tolist(),
    tokenizer
)
val_dataset = ClaimDataset(
    val_df['claim1'].tolist(),
    val_df['label_id'].tolist(),
    tokenizer
)
test_dataset = ClaimDataset(
    test_df['claim1'].tolist(),
    test_df['label_id'].tolist(),
    tokenizer
)

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Model Setup
# ============================================================
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)
model.to(device)

EPOCHS = 4
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Training + Validation Loop
# ============================================================
def evaluate(model, loader):
    model.eval()
    preds, true_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(true_labels, preds)
    return avg_loss, acc, preds, true_labels

best_val_acc = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    start = time.time()

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        if step % 20 == 0:
            print(f"  Epoch {epoch+1} Step {step}/{len(train_loader)} Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    elapsed = time.time() - start
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"  Time: {elapsed:.0f}s\n")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained('/content/drive/MyDrive/experiment1/best_model')
        tokenizer.save_pretrained('/content/drive/MyDrive/experiment1/best_model')
        print(f"  Saved best model (val_acc: {val_acc:.4f})\n")

# Test Evaluation
# ============================================================
# Load best model
best_model = RobertaForSequenceClassification.from_pretrained(
    '/content/drive/MyDrive/experiment1/best_model'
)
best_model.to(device)

test_loss, test_acc, test_preds, test_true = evaluate(best_model, test_loader)

print(f"Test Accuracy: {test_acc:.4f}\n")
print("Classification Report:")
print(classification_report(
    test_true, test_preds,
    target_names=['human', 'ai_generated', 'refine']
))
print("Confusion Matrix:")
print(confusion_matrix(test_true, test_preds))

loss and accuracy chart

#loss pic
import matplotlib.pyplot as plt

epochs = [1, 2, 3, 4]
train_loss = [0.6918, 0.4749, 0.3147, 0.2026]
val_loss = [0.4941, 0.4232, 0.2468, 0.2423]
val_acc = [0.6622, 0.7622, 0.9044, 0.9089]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(epochs, train_loss, 'b-o', label='Train Loss')
ax1.plot(epochs, val_loss, 'r-o', label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Train vs Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy curve
ax2.plot(epochs, val_acc, 'g-o', label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/experiment1/training_curves.png', dpi=150)
plt.show()